# 말투 AI — 상대방 말투 파인튜닝 (Unsloth · QLoRA · 무료 T4)

카카오톡 대화에서 **한 사람의 말투**를 배우는 LoRA를 학습하고, 휴대폰에서 돌릴 **GGUF(Q4_K_M)** 파일로 내보냅니다.

- 실행 환경: **Kaggle**(설정 → Accelerator: *GPU T4 x1*, Internet: *On*) 또는 **Google Colab**(런타임 → T4 GPU)
- 입력: 앱 ① 탭의 **"학습 데이터 내려받기"** 로 만든 `persona_train.jsonl` (또는 `training/prepare_kakao.py` 결과)
- 출력: `persona-q4_k_m.gguf` → 휴대폰으로 옮겨 앱 ② 탭 **"이 기기의 GGUF 선택"**

> ⚠️ 이 노트북에 올린 데이터는 Kaggle/Google 서버로 전송됩니다. 데이터셋은 **비공개(Private)** 로 올리고, 학습이 끝나면 데이터셋·출력 파일을 삭제하세요.
> 학습된 모델에도 대화 내용이 일부 기억될 수 있으니 **모델 파일을 공개 저장소에 올리지 마세요.**

## 1. 설치 (약 2~3분)

In [ ]:
%%capture
!pip install -q unsloth

## 2. 설정
- `BASE_MODEL`: 아이폰 웹은 **1.5B**(약 1GB) 권장, Android 앱(RAM 8GB↑)은 **3B**(약 2GB)도 가능
- `PERSONA_NAME`: 앱 ② 탭의 "따라할 이름"과 **반드시 같아야** 합니다(시스템 프롬프트에 들어감). 앱에서 내보낸 JSONL을 쓰면 자동으로 읽습니다.

In [ ]:
import glob, json, os

BASE_MODEL = "unsloth/Qwen2.5-1.5B-Instruct"   # 또는 "unsloth/Qwen2.5-3B-Instruct", "unsloth/Qwen2.5-0.5B-Instruct"
MAX_SEQ_LEN = 1024
EPOCHS = 2                 # 답장 수천 개 기준. 적으면 3, 수만 개면 1
LORA_RANK = 16
QUANT = "q4_k_m"
OUT_NAME = "persona"

# Kaggle: /kaggle/input/<데이터셋>/persona_train.jsonl · Colab: 왼쪽 파일 패널로 업로드
candidates = glob.glob("/kaggle/input/**/*.jsonl", recursive=True) + glob.glob("/content/*.jsonl") + glob.glob("*.jsonl")
DATA_PATH = candidates[0] if candidates else None
print("data:", DATA_PATH)
assert DATA_PATH, "persona_train.jsonl 을 업로드하세요 (Kaggle: Add Data → 비공개 데이터셋, Colab: 파일 패널)."

rows = [json.loads(l) for l in open(DATA_PATH, encoding="utf-8") if l.strip()]
SYSTEM_PROMPT = rows[0]["messages"][0]["content"]
replies = sum(1 for r in rows for m in r["messages"] if m["role"] == "assistant")
print(f"samples={len(rows)} replies={replies}")
print("system prompt:", SYSTEM_PROMPT)
if replies < 300:
    print("⚠️ 답장이 300개 미만이면 말투가 잘 안 배워집니다(1,000개 이상 권장).")

## 3. 모델 불러오기 (4bit) + LoRA

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LEN,
    load_in_4bit=True,
)
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_RANK,
    lora_alpha=LORA_RANK * 2,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

## 4. 데이터 → Qwen 채팅 형식

In [ ]:
from datasets import Dataset
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(tokenizer, chat_template="qwen-2.5")

def to_text(batch):
    return {"text": [tokenizer.apply_chat_template(m, tokenize=False, add_generation_prompt=False) for m in batch["messages"]]}

ds = Dataset.from_list(rows).map(to_text, batched=True, remove_columns=["messages"])
ds = ds.filter(lambda x: len(tokenizer(x["text"]).input_ids) <= MAX_SEQ_LEN)
split = ds.train_test_split(test_size=0.05, seed=3407) if len(ds) >= 200 else None
train_ds, eval_ds = (split["train"], split["test"]) if split else (ds, None)
print(train_ds, eval_ds)
print(train_ds[0]["text"][:600])

## 5. 학습 — 상대방(assistant) 답장 부분만 학습

In [ ]:
from trl import SFTConfig, SFTTrainer
from unsloth import is_bfloat16_supported
from unsloth.chat_templates import train_on_responses_only

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    args=SFTConfig(
        dataset_text_field="text",
        max_seq_length=MAX_SEQ_LEN,
        per_device_train_batch_size=4,
        gradient_accumulation_steps=4,
        num_train_epochs=EPOCHS,
        learning_rate=2e-4,
        warmup_ratio=0.03,
        lr_scheduler_type="cosine",
        weight_decay=0.01,
        logging_steps=10,
        eval_strategy="epoch" if eval_ds is not None else "no",
        save_strategy="no",
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        optim="adamw_8bit",
        seed=3407,
        output_dir="outputs",
        report_to="none",
    ),
)
trainer = train_on_responses_only(
    trainer,
    instruction_part="<|im_start|>user\n",
    response_part="<|im_start|>assistant\n",
)
stats = trainer.train()
print(stats)

## 6. 바로 시험해 보기

In [ ]:
FastLanguageModel.for_inference(model)

def reply(text, history=()):
    msgs = [{"role": "system", "content": SYSTEM_PROMPT}, *history, {"role": "user", "content": text}]
    ids = tokenizer.apply_chat_template(msgs, add_generation_prompt=True, return_tensors="pt").to("cuda")
    out = model.generate(input_ids=ids, max_new_tokens=80, temperature=0.8, top_p=0.9, do_sample=True)
    return tokenizer.decode(out[0][ids.shape[1]:], skip_special_tokens=True)

for q in ["뭐해?", "오늘 저녁 뭐 먹을까", "주말에 시간 돼?"]:
    print(q, "→", reply(q))

## 7. GGUF(Q4_K_M)로 내보내기 (약 5~10분)
2GB를 넘으면 웹(wllama)에서 한 파일로 못 읽으므로 512MB 단위로도 나눠 둡니다(Android/iPhone 모두 나눈 파일을 **함께 선택**하면 됩니다).

In [ ]:
model.save_pretrained_gguf(OUT_NAME, tokenizer, quantization_method=QUANT)

import shutil, subprocess
ggufs = sorted(glob.glob(f"{OUT_NAME}*/*.gguf") + glob.glob(f"{OUT_NAME}*.gguf"))
print(ggufs)
final = [g for g in ggufs if QUANT.upper() in g.upper() or QUANT in g][-1]
os.makedirs("export", exist_ok=True)
target = f"export/{OUT_NAME}-{QUANT}.gguf"
shutil.copy(final, target)
size = os.path.getsize(target)
print(f"{target}: {size/1e9:.2f} GB")

split_bin = (glob.glob("llama.cpp/**/llama-gguf-split", recursive=True) + glob.glob(os.path.expanduser("~/.unsloth/llama.cpp/**/llama-gguf-split"), recursive=True))
if split_bin and size > 512 * 1024 * 1024:
    subprocess.run([split_bin[0], "--split-max-size", "512M", target, f"export/{OUT_NAME}-{QUANT}"], check=True)
print(sorted(os.listdir("export")))

## 8. 내려받기
- **Kaggle**: 오른쪽 *Output* 패널 → `export/` 파일 다운로드 (또는 *Save Version* 후 Output 탭)
- **Colab**: 아래 셀 실행

받은 `.gguf`를 휴대폰으로 옮긴 뒤(USB·AirDrop·내 파일), 앱 ② 탭에서 **"파인튜닝한 내 모델"** → **"이 기기의 GGUF 선택"**.
따라할 이름은 위에 출력된 시스템 프롬프트 속 이름과 같게 입력하세요.

In [ ]:
try:
    from google.colab import files
    for f in sorted(os.listdir("export")):
        files.download(f"export/{f}")
except ImportError:
    print("Kaggle: Output 패널에서 export/ 폴더를 내려받으세요.")

## 9. 정리 (개인정보)
학습이 끝나면 업로드한 데이터셋과 이 노트북 출력물을 삭제하세요. 모델 파일을 Hugging Face 등에 올려야 한다면 **반드시 Private** 저장소를 쓰세요.

In [ ]:
# 런타임 안의 학습 데이터와 체크포인트 삭제
for p in ["outputs", OUT_NAME, f"{OUT_NAME}_gguf"]:
    shutil.rmtree(p, ignore_errors=True)
if DATA_PATH and not DATA_PATH.startswith("/kaggle/input"):
    os.remove(DATA_PATH)
print("cleaned (Kaggle 입력 데이터셋은 Kaggle 사이트에서 직접 삭제하세요)")